# Case Study 3: Manufacturing Process Signal Feature Extraction

This notebook uses a simulated CNC power signal to show how raw sensor data can be segmented into windows and converted into features for anomaly detection.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['figure.figsize'] = (10, 4)


## Task 1: Load and visualize the process signal

In [ ]:
df = pd.read_csv('../data/case_study_3_cnc_power_signal.csv')
df.head()

In [ ]:
plt.plot(df['time_s'], df['power_W'])
plt.xlabel('Time (s)')
plt.ylabel('Power (W)')
plt.title('CNC Power Signal')
plt.show()

## Task 2: Segment the signal into windows
Each row in the feature table will summarize one short time window.

In [ ]:
window_seconds = 10
fs = 10
window_size = window_seconds * fs
rows = []
for start in range(0, len(df)-window_size+1, window_size):
    segment = df.iloc[start:start+window_size]
    x = segment['power_W'].values
    rows.append({
        'start_time_s': segment['time_s'].iloc[0],
        'mean_power': x.mean(),
        'std_power': x.std(),
        'rms_power': np.sqrt(np.mean(x**2)),
        'peak_power': x.max(),
        'energy_proxy': np.sum(x) / fs,
        'anomaly_label': int(segment['anomaly_label'].max())
    })
features = pd.DataFrame(rows)
features.head()

## Task 3: Visualize feature behavior

In [ ]:
plt.plot(features['start_time_s'], features['mean_power'], label='Mean power')
plt.plot(features['start_time_s'], features['std_power'], label='Std power')
plt.xlabel('Window start time (s)')
plt.ylabel('Feature value')
plt.legend()
plt.title('Window-Level Features')
plt.show()

## Task 4: Use Isolation Forest on extracted features

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
feature_cols = ['mean_power','std_power','rms_power','peak_power','energy_proxy']
model = IsolationForest(contamination=0.12, random_state=42)
features['iforest_pred'] = model.fit_predict(features[feature_cols])
features['detected_anomaly'] = (features['iforest_pred'] == -1).astype(int)
print(confusion_matrix(features['anomaly_label'], features['detected_anomaly']))
print(classification_report(features['anomaly_label'], features['detected_anomaly'], zero_division=0))

## Reflection
1. Which feature seems most sensitive to abnormal behavior?
2. Why might a power signal increase during tool wear?
3. Why should an engineer inspect the detected windows rather than accept the model blindly?